[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/probability_statistics/10_bayesian_inference/first_principles.ipynb)

# Topic 10: Bayesian Inference

## 1. First-Principles Intuition & Motivation

Two questions look similar but are not: "how likely are these data if the parameter is $\theta$?" and "how plausible is $\theta$ now that I have seen these data?" The first is the likelihood and is answered by the model. The second requires probability *over parameters*, and the only coherent way to obtain it is Bayes' theorem:

$$
p(\theta \mid x) = \frac{p(x \mid \theta)\,p(\theta)}{p(x)}, \qquad p(x) = \int p(x\mid\theta)\,p(\theta)\,d\theta .
$$

The prior $p(\theta)$ is the price of admission — you cannot get a probability distribution over $\theta$ out of a model that only describes $x$. In exchange you receive an object that answers every question directly: point estimates are posterior functionals, intervals are posterior probabilities, predictions are posterior averages, and decisions minimize posterior expected loss.

The mechanics are best read in log space, where the multiplication becomes addition:

$$
\ln p(\theta\mid x) = \underbrace{\ell(\theta)}_{\text{evidence from data}} + \underbrace{\ln p(\theta)}_{\text{evidence from before}} - \underbrace{\ln p(x)}_{\text{constant in } \theta} .
$$

Because $\ell$ accumulates one term per observation while $\ln p(\theta)$ is fixed, the data eventually dominate; the prior matters exactly in proportion to how little data you have.

### 1.1 Why Coherence Forces the Bayesian Update

Suppose you want to update beliefs about $\theta$ upon seeing $x$ and you insist on two properties: the answer depends only on the observed $x$ (not on data you might have seen), and beliefs about compound events respect the product and sum rules of probability. Cox's theorem and the Dutch-book arguments of de Finetti both conclude that the update must be

$$
\text{posterior} \ \propto\ \text{likelihood}\times\text{prior} .
$$

Any other rule allows a set of bets that loses money with certainty. This is why Bayes' rule is not one estimation technique among many but the unique coherent bookkeeping for uncertainty — the same reason the sum-to-one axiom is not negotiable.

A second structural virtue is **sequential consistency**. Splitting data into $x_{1:m}$ and $x_{m+1:n}$,

$$
p\left(\theta\mid x_{1:n}\right) \ \propto\ p\left(x_{m+1:n}\mid\theta\right)\,p\left(\theta\mid x_{1:m}\right) ,
$$

so processing data in one batch, in two batches, or one point at a time gives identical answers. Kalman filters, online learning, and streaming A/B tests all exploit this.

### 1.2 The Prior Is a Pile of Pseudo-Observations

Abstract priors become concrete inside conjugate families, where the posterior stays in the same family and updating amounts to adding counts. A $\text{Beta}(\alpha,\beta)$ prior for a coin behaves exactly like $\alpha$ prior heads and $\beta$ prior tails:

$$
\text{Beta}(\alpha,\beta) \ +\ (k \text{ heads}, n-k \text{ tails}) \ \longrightarrow\ \text{Beta}(\alpha+k,\ \beta+n-k) .
$$

Its mean, $\frac{\alpha+k}{\alpha+\beta+n}$, is a weighted average of the prior mean and the sample proportion with weights $\frac{\alpha+\beta}{\alpha+\beta+n}$ and $\frac{n}{\alpha+\beta+n}$. The quantity $\alpha+\beta$ is the *prior sample size*: it says how many observations of evidence your prior is worth. Choosing a prior then becomes an answerable engineering question rather than a philosophical one — how much data would I need to see before I would abandon my current belief?

### 1.3 Uncertainty Must Be Propagated, Not Collapsed

A trained model that reports $p(\tilde x\mid\hat\theta)$ is pretending that $\hat\theta$ is exactly right. The Bayesian alternative averages over what $\theta$ could be:

$$
p(\tilde x\mid x) = \int p(\tilde x\mid\theta)\,p(\theta\mid x)\,d\theta .
$$

The result is strictly wider than the plug-in version, by the law of total variance:

$$
\operatorname{Var}\left(\tilde x\mid x\right) = \underbrace{E\left[\operatorname{Var}(\tilde x\mid\theta)\,\middle|\,x\right]}_{\text{aleatoric: irreducible noise}} + \underbrace{\operatorname{Var}\left(E[\tilde x\mid\theta]\,\middle|\,x\right)}_{\text{epistemic: parameter uncertainty}} .
$$

The second term is exactly what plug-in prediction discards, and it is the term that shrinks with more data. The decomposition is the mathematical statement of the distinction between "the world is noisy" and "I don't know the world yet" — the basis of active learning, exploration bonuses, and out-of-distribution detection.

## 2. Rigorous Mathematical Definitions & Theorem Statements

**Theorem 2.1 (Bayes' theorem for densities).** Let $\theta$ have prior density $p(\theta)$ on $\Theta$ and let $x$ have conditional density $p(x\mid\theta)$. Then, wherever $p(x) \gt 0$,

$$
p(\theta\mid x) = \frac{p(x\mid\theta)\,p(\theta)}{p(x)}, \qquad p(x) = \int_{\Theta} p(x\mid\theta)\,p(\theta)\,d\theta .
$$

**Definition 2.2 (Marginal likelihood / evidence).** $p(x)$ above; also written $p(x\mid M)$ when comparing models $M$. It is the prior-predictive density of the observed data.

**Definition 2.3 (Posterior predictive).** For new data $\tilde{x}$ conditionally independent of $x$ given $\theta$,

$$
p(\tilde x\mid x) = \int_\Theta p(\tilde x\mid\theta)\,p(\theta\mid x)\,d\theta .
$$

**Definition 2.4 (Credible set).** A set $C \subseteq \Theta$ is a $(1-\alpha)$ *credible set* if

$$
P\left(\theta\in C\mid x\right) = \int_C p(\theta\mid x)\,d\theta = 1-\alpha .
$$

The **highest posterior density (HPD)** region is the credible set $\{\theta : p(\theta\mid x) \ge c\}$ with $c$ chosen to reach level $1-\alpha$; it is the shortest such set and is invariant only up to the same Jacobian caveat that affects the mode.

**Definition 2.5 (Conjugate family).** A family $\mathcal{P}$ of priors is *conjugate* to a likelihood $p(x\mid\theta)$ if $p(\theta)\in\mathcal{P}$ implies $p(\theta\mid x)\in\mathcal{P}$ for all $x$.

**Theorem 2.6 (Exponential-family conjugacy).** If $p(x\mid\eta) = h(x)\exp\left(\eta^{\top}T(x) - A(\eta)\right)$, then the family

$$
p(\eta\mid \chi, \nu) \ \propto\ \exp\left(\eta^{\top}\chi - \nu A(\eta)\right)
$$

is conjugate, with update $(\chi,\nu) \mapsto \left(\chi + \sum_i T(x_i),\ \nu + n\right)$. Every standard conjugate pair is an instance: the hyperparameters are pseudo-sufficient-statistics and a pseudo-count.

**Proposition 2.7 (The four workhorse conjugate pairs).**

| Likelihood | Prior | Posterior |
|---|---|---|
| $\text{Binomial}(n,p)$ | $\text{Beta}(\alpha,\beta)$ | $\text{Beta}(\alpha+k,\ \beta+n-k)$ |
| $\mathcal{N}(\mu,\sigma^2)$, $\sigma^2$ known | $\mathcal{N}(\mu_0,\tau_0^2)$ | $\mathcal{N}(\mu_n,\tau_n^2)$ with precisions adding |
| $\text{Poisson}(\lambda)$ | $\text{Gamma}(a,b)$ | $\text{Gamma}\left(a+\sum_i x_i,\ b+n\right)$ |
| $\text{Categorical}(p)$ | $\text{Dirichlet}(\alpha)$ | $\text{Dirichlet}(\alpha + \text{counts})$ |

**Definition 2.8 (Jeffreys prior).** $p_J(\theta) \propto \sqrt{\det I(\theta)}$, where $I$ is the Fisher information. It is invariant under reparameterization: if $\phi = g(\theta)$ then the Jeffreys prior computed in $\phi$ equals the pushforward of the one computed in $\theta$.

**Theorem 2.9 (Bayes estimators and loss).** The action minimizing posterior expected loss $E\left[L(\theta,a)\mid x\right]$ is

- the **posterior mean** for squared loss $L = (\theta-a)^2$;
- the **posterior median** for absolute loss $L = \vert \theta - a\vert$;
- the **posterior mode** for $0$–$1$ loss $L = \mathbf{1}\{\vert \theta - a\vert \gt \epsilon\}$ as $\epsilon\to 0$.

**Definition 2.10 (Bayes factor).** For models $M_1, M_2$,

$$
\text{BF}_{12} = \frac{p(x\mid M_1)}{p(x\mid M_2)}, \qquad \frac{P(M_1\mid x)}{P(M_2\mid x)} = \text{BF}_{12}\cdot\frac{P(M_1)}{P(M_2)} .
$$

**Theorem 2.11 (Bernstein–von Mises).** Under regularity conditions (identifiability, smoothness, a prior positive and continuous at $\theta_0$), the posterior satisfies, in total variation,

$$
\left\Vert\, p\left(\theta\mid x_{1:n}\right) - \mathcal{N}\left(\hat\theta_n,\ \left[nI(\theta_0)\right]^{-1}\right)\,\right\Vert_{\text{TV}} \xrightarrow{P} 0 .
$$

Consequently Bayesian credible sets are asymptotically frequentist confidence sets, and the prior's influence vanishes.

**Definition 2.12 (Evidence lower bound, ELBO).** For any distribution $q(\theta)$,

$$
\ln p(x) = \underbrace{E_q\left[\ln\frac{p(x,\theta)}{q(\theta)}\right]}_{\mathcal{L}(q)\ =\ \text{ELBO}} + D_{\text{KL}}\left(q(\theta)\,\Vert\,p(\theta\mid x)\right) \ \ge\ \mathcal{L}(q) .
$$

Maximizing the ELBO over a tractable family $\mathcal{Q}$ is **variational inference**; the gap is exactly the KL divergence to the true posterior.

**Theorem 2.13 (Metropolis–Hastings correctness).** Let $q(\theta'\mid\theta)$ be a proposal and accept with probability

$$
a\left(\theta\to\theta'\right) = \min\left\{1,\ \frac{\tilde{p}(\theta')\,q(\theta\mid\theta')}{\tilde{p}(\theta)\,q(\theta'\mid\theta)}\right\} ,
$$

where $\tilde p \propto p(\theta\mid x)$ need only be known up to a constant. The resulting chain satisfies detailed balance with respect to $p(\theta\mid x)$; if it is irreducible and aperiodic, its ergodic averages converge to posterior expectations.

## 3. Step-by-Step Mathematical Proofs & Derivations

### Proof 3.1: Bayes' Theorem and Sequential Consistency

**Claim.** $p(\theta\mid x) = p(x\mid\theta)p(\theta)/p(x)$, and updating on $x_{1:n}$ at once equals updating on $x_{1:m}$ then on $x_{m+1:n}$.

**Proof (theorem).** By the definition of conditional density, the joint factors two ways:

$$
p(\theta, x) = p(\theta\mid x)\,p(x) = p(x\mid\theta)\,p(\theta) .
$$

Dividing by $p(x) \gt 0$ gives the result. The denominator is fixed by requiring $\int p(\theta\mid x)d\theta = 1$:

$$
p(x) = \int p(x\mid\theta)p(\theta)\,d\theta ,
$$

which is why $p(x)$ can be ignored during optimization or MCMC and must be computed only for model comparison. $\square$

**Proof (sequential consistency).** With conditional independence of observations given $\theta$,

$$
p\left(\theta\mid x_{1:n}\right) \ \propto\ p\left(x_{1:n}\mid\theta\right)p(\theta) = p\left(x_{m+1:n}\mid\theta\right)\underbrace{p\left(x_{1:m}\mid\theta\right)p(\theta)}_{\propto\ p(\theta\mid x_{1:m})} \ \propto\ p\left(x_{m+1:n}\mid\theta\right)\,p\left(\theta\mid x_{1:m}\right) .
$$

Both sides are normalized densities in $\theta$, and proportional normalized densities are equal. $\blacksquare$

**Consequences.** Order does not matter (exchangeability of the update), streaming and batch inference agree exactly, and "yesterday's posterior is today's prior" is a theorem rather than a slogan. Filtering algorithms simply alternate this update with a prediction step.

### Proof 3.2: Beta–Binomial Conjugacy and Laplace's Rule of Succession

**Claim.** If $p\sim\text{Beta}(\alpha,\beta)$ and $k \sim \text{Binomial}(n,p)$, then $p\mid k \sim \text{Beta}(\alpha+k,\ \beta+n-k)$.

**Proof.** The prior density is

$$
p(p) = \frac{p^{\alpha-1}(1-p)^{\beta-1}}{B(\alpha,\beta)}, \qquad B(\alpha,\beta) = \frac{\Gamma(\alpha)\Gamma(\beta)}{\Gamma(\alpha+\beta)} ,
$$

and the likelihood is $\binom{n}{k}p^k(1-p)^{n-k}$. Multiplying and dropping every factor free of $p$:

$$
p(p\mid k)\ \propto\ p^{k}(1-p)^{n-k}\cdot p^{\alpha-1}(1-p)^{\beta-1} = p^{(\alpha+k)-1}(1-p)^{(\beta+n-k)-1} .
$$

This is the kernel of a $\text{Beta}(\alpha+k,\ \beta+n-k)$ density, and since a density is determined by its kernel, the normalizing constant must be $1/B(\alpha+k,\beta+n-k)$. $\blacksquare$

**Posterior summaries.** With $a = \alpha+k$, $b = \beta+n-k$:

$$
E[p\mid k] = \frac{a}{a+b} = \frac{\alpha+k}{\alpha+\beta+n}, \qquad \text{mode} = \frac{a-1}{a+b-2}, \qquad \operatorname{Var}(p\mid k) = \frac{ab}{(a+b)^2(a+b+1)} .
$$

**Shrinkage decomposition.** Write $m = \alpha+\beta$ (prior sample size) and $\mu_0 = \alpha/m$ (prior mean):

$$
E[p\mid k] = \frac{m}{m+n}\,\mu_0 + \frac{n}{m+n}\cdot\frac{k}{n} ,
$$

an explicit convex combination of prior mean and MLE. The posterior variance decays like $1/n$, matching the frequentist $p(1-p)/n$.

**Rule of succession.** With a uniform prior $\text{Beta}(1,1)$ and $k$ successes in $n$ trials, the posterior predictive probability of a success on the next trial is

$$
P\left(\tilde x = 1\mid k\right) = \int_0^1 p\cdot p(p\mid k)\,dp = E[p\mid k] = \frac{k+1}{n+2} .
$$

Laplace's answer to "the sun has risen $n$ times, will it rise tomorrow?": never $1$, always strictly inside $(0,1)$, and approaching $k/n$ at rate $O(1/n)$. Compare with the MLE's $k/n$, which asserts impossibility after any run of one outcome.

### Proof 3.3: Normal–Normal Conjugacy — Precisions Add

**Claim.** If $\mu\sim\mathcal{N}(\mu_0,\tau_0^2)$ and $x_1,\ldots,x_n\mid\mu \sim \mathcal{N}(\mu,\sigma^2)$ i.i.d. with $\sigma^2$ known, then

$$
\mu\mid x \ \sim\ \mathcal{N}\left(\mu_n, \tau_n^2\right), \qquad \frac{1}{\tau_n^2} = \frac{1}{\tau_0^2} + \frac{n}{\sigma^2}, \qquad \mu_n = \tau_n^2\left(\frac{\mu_0}{\tau_0^2} + \frac{n\bar x}{\sigma^2}\right) .
$$

**Proof.** Work with the log-posterior and discard $\mu$-free terms:

$$
\ln p(\mu\mid x) = -\frac{1}{2\sigma^2}\sum_{i=1}^n (x_i-\mu)^2 - \frac{(\mu-\mu_0)^2}{2\tau_0^2} + C .
$$

Expand the sum using $\sum_i (x_i - \mu)^2 = \sum_i(x_i-\bar x)^2 + n(\bar x - \mu)^2$; the first piece is $\mu$-free. Collecting powers of $\mu$:

$$
\ln p(\mu\mid x) = -\frac12\left[\underbrace{\left(\frac{n}{\sigma^2}+\frac{1}{\tau_0^2}\right)}_{=\,1/\tau_n^2}\mu^2 - 2\underbrace{\left(\frac{n\bar x}{\sigma^2}+\frac{\mu_0}{\tau_0^2}\right)}_{=\,\mu_n/\tau_n^2}\mu\right] + C' .
$$

Completing the square gives $-\frac{(\mu-\mu_n)^2}{2\tau_n^2} + C''$, the log-kernel of $\mathcal{N}(\mu_n,\tau_n^2)$ with the stated parameters. $\blacksquare$

**Precision reading.** Defining precision $\lambda = 1/\text{variance}$, the update is startlingly simple:

$$
\lambda_n = \lambda_0 + n\lambda_{\text{data}}, \qquad \mu_n = \frac{\lambda_0\mu_0 + n\lambda_{\text{data}}\bar x}{\lambda_0 + n\lambda_{\text{data}}} .
$$

Precisions add; means combine as a precision-weighted average. This is *exactly* the Kalman filter measurement update, and exactly the inverse-variance weighting used to combine independent experimental measurements in physics.

**Limits.** As $\tau_0^2\to\infty$ (flat prior) we get $\mu_n \to \bar x$ and $\tau_n^2 \to \sigma^2/n$ — the frequentist answer. As $n\to\infty$ with $\tau_0$ fixed the same limit holds, illustrating Bernstein–von Mises concretely.

**Posterior predictive.** For a new observation $\tilde x \sim \mathcal{N}(\mu,\sigma^2)$, independence lets the variances add:

$$
\tilde x \mid x \ \sim\ \mathcal{N}\left(\mu_n,\ \sigma^2 + \tau_n^2\right) ,
$$

visibly wider than the plug-in $\mathcal{N}(\mu_n,\sigma^2)$ by exactly the epistemic term $\tau_n^2$.

### Proof 3.4: Gamma–Poisson Conjugacy and the Negative-Binomial Predictive

**Claim.** If $\lambda\sim\text{Gamma}(a,b)$ (shape $a$, rate $b$) and $x_1,\ldots,x_n\mid\lambda\sim\text{Poisson}(\lambda)$, then $\lambda\mid x\sim\text{Gamma}\left(a+\sum_i x_i,\ b+n\right)$, and the posterior predictive is negative binomial.

**Proof (conjugacy).** The prior kernel is $\lambda^{a-1}e^{-b\lambda}$ and the likelihood kernel is $\lambda^{\sum_i x_i}e^{-n\lambda}$. Multiplying,

$$
p(\lambda\mid x)\ \propto\ \lambda^{a + \sum_i x_i - 1}\,e^{-(b+n)\lambda} ,
$$

the kernel of $\text{Gamma}\left(a+\sum_i x_i,\ b+n\right)$. $\square$

The hyperparameters read as pseudo-data: $a$ prior events observed in $b$ prior units of exposure. The posterior mean is

$$
E[\lambda\mid x] = \frac{a+\sum_i x_i}{b+n} = \frac{b}{b+n}\cdot\frac{a}{b} + \frac{n}{b+n}\cdot\bar{x} ,
$$

again a shrinkage of the MLE $\bar x$ toward the prior mean $a/b$.

**Proof (predictive).** Integrate the Poisson likelihood against the posterior $\text{Gamma}(a', b')$ with $a' = a+\sum_i x_i$, $b' = b+n$:

$$
p(\tilde x\mid x) = \int_0^\infty \frac{\lambda^{\tilde x}e^{-\lambda}}{\tilde x!}\cdot\frac{b'^{a'}}{\Gamma(a')}\lambda^{a'-1}e^{-b'\lambda}\,d\lambda = \frac{b'^{a'}}{\tilde x!\,\Gamma(a')}\int_0^\infty \lambda^{a'+\tilde x-1}e^{-(b'+1)\lambda}d\lambda .
$$

The integral is a Gamma normalizer, $\Gamma(a'+\tilde x)/(b'+1)^{a'+\tilde x}$, so

$$
p(\tilde x\mid x) = \frac{\Gamma(a'+\tilde x)}{\tilde x!\,\Gamma(a')}\left(\frac{b'}{b'+1}\right)^{a'}\left(\frac{1}{b'+1}\right)^{\tilde x} ,
$$

which is the negative binomial law $\text{NB}\left(a',\ \frac{b'}{b'+1}\right)$. $\blacksquare$

**Overdispersion.** The Poisson forces $\operatorname{Var} = E$; the negative binomial has

$$
\operatorname{Var}(\tilde x\mid x) = \frac{a'}{b'}\left(1 + \frac{1}{b'}\right) \gt E[\tilde x\mid x] = \frac{a'}{b'} .
$$

Marginalizing parameter uncertainty *creates* overdispersion — the single most common reason count data in the wild are negative-binomial rather than Poisson.

### Proof 3.5: Bayesian Decision Theory — Which Summary to Report

**Claim.** The posterior mean minimizes expected squared loss, the median minimizes expected absolute loss, and the mode is the limit of $0$–$1$ loss.

**Proof (squared loss).** Let $\rho(a) = E\left[(\theta-a)^2\mid x\right]$. Expand:

$$
\rho(a) = E\left[\theta^2\mid x\right] - 2a\,E[\theta\mid x] + a^2 .
$$

Differentiating, $\rho'(a) = -2E[\theta\mid x] + 2a = 0$ gives $a^\star = E[\theta\mid x]$, and $\rho''(a) = 2 \gt 0$ confirms a minimum. Moreover $\rho(a^\star) = \operatorname{Var}(\theta\mid x)$: the achievable risk is the posterior variance. $\square$

**Proof (absolute loss).** Let $\rho(a) = E\left[\vert \theta-a\vert\ \middle|\ x\right] = \int_{-\infty}^a (a-\theta)p\,d\theta + \int_a^\infty(\theta-a)p\,d\theta$. Differentiating under the integral (Leibniz; boundary terms cancel):

$$
\rho'(a) = \int_{-\infty}^a p(\theta\mid x)d\theta - \int_a^\infty p(\theta\mid x)d\theta = 2F(a\mid x) - 1 .
$$

Setting this to zero gives $F(a^\star\mid x) = 1/2$: the posterior median. Since $\rho'$ is non-decreasing, $\rho$ is convex and the stationary point is the global minimum. $\square$

**Proof ($0$–$1$ loss).** With $L_\epsilon(\theta,a) = \mathbf{1}\{\vert\theta-a\vert \gt \epsilon\}$,

$$
E\left[L_\epsilon\mid x\right] = 1 - P\left(\vert\theta-a\vert\le\epsilon\mid x\right) = 1 - \int_{a-\epsilon}^{a+\epsilon}p(\theta\mid x)d\theta \approx 1 - 2\epsilon\,p(a\mid x)
$$

for small $\epsilon$ and continuous $p$. Minimizing over $a$ maximizes $p(a\mid x)$: the posterior mode, i.e. MAP. $\blacksquare$

**Reading.** The choice among mean, median, and mode is not a matter of taste but of the loss you actually face. Asymmetric losses give quantile estimators: for the pinball loss with weight $\tau$ on under-prediction, the optimum is the posterior $\tau$-quantile — the fact behind quantile regression and inventory (newsvendor) decisions. When the posterior is symmetric and unimodal all three coincide, which is why the distinction is invisible in Gaussian textbook examples and painfully visible in skewed real ones.

### Proof 3.6: Bernstein–von Mises — Why the Prior Washes Out

**Claim (sketch with the key steps).** Under regularity, $p(\theta\mid x_{1:n}) \approx \mathcal{N}\left(\hat\theta_n, \left[nI(\theta_0)\right]^{-1}\right)$ in total variation.

**Step 1 — expand the log-posterior.** Write

$$
\ln p(\theta\mid x_{1:n}) = \ell_n(\theta) + \ln p(\theta) + C .
$$

Taylor-expand $\ell_n$ about the MLE $\hat\theta_n$, where the score vanishes:

$$
\ell_n(\theta) = \ell_n(\hat\theta_n) - \frac{n}{2}\left(\theta-\hat\theta_n\right)^{\top}\hat{I}_n\left(\theta-\hat\theta_n\right) + R_n, \qquad \hat I_n = -\frac1n\nabla^2\ell_n(\hat\theta_n) .
$$

**Step 2 — the prior is negligible at the posterior's scale.** The posterior concentrates in a ball of radius $O(n^{-1/2})$ around $\hat\theta_n$. On that shrinking ball a prior that is continuous and positive at $\theta_0$ satisfies

$$
\ln p(\theta) = \ln p(\theta_0) + O\left(n^{-1/2}\right) ,
$$

i.e. it is asymptotically *constant* over the region carrying all the posterior mass. It therefore cannot influence the shape — only quadratic and higher terms scaled by $n$ survive.

**Step 3 — identify the Gaussian.** Exponentiating what remains,

$$
p(\theta\mid x_{1:n})\ \propto\ \exp\left(-\frac{n}{2}\left(\theta-\hat\theta_n\right)^{\top}\hat I_n \left(\theta-\hat\theta_n\right)\right)\left(1+o(1)\right) ,
$$

the kernel of $\mathcal{N}\left(\hat\theta_n, (n\hat I_n)^{-1}\right)$, and $\hat I_n\xrightarrow{P} I(\theta_0)$ by the LLN. Control of $R_n$ and of the tails (where the expansion is invalid but the posterior mass is exponentially small) upgrades this to total-variation convergence. $\blacksquare$

**Consequences.**

1. **Prior insensitivity.** Any two priors positive and continuous at $\theta_0$ give posteriors that merge; disagreement is an $O(n^{-1/2})$ phenomenon.
2. **Frequentist–Bayesian reconciliation.** Credible intervals have asymptotically correct frequentist coverage, and posterior standard deviations match MLE standard errors.
3. **The hypotheses matter.** BvM fails when the prior assigns zero mass near $\theta_0$ (no amount of data recovers), on boundary problems, in non-regular models such as $\text{Unif}(0,\theta)$, and in high-dimensional regimes where $d$ grows with $n$ — exactly the regimes where Bayesian and frequentist answers genuinely differ.

### Proof 3.7: The ELBO and Variational Inference

**Claim.** For any density $q(\theta)$ with support containing that of the posterior,

$$
\ln p(x) = \mathcal{L}(q) + D_{\text{KL}}\left(q\,\Vert\,p(\cdot\mid x)\right), \qquad \mathcal{L}(q) = E_q\left[\ln p(x,\theta)\right] - E_q\left[\ln q(\theta)\right] .
$$

**Proof.** Insert $q$ and split the logarithm:

$$
D_{\text{KL}}\left(q\,\Vert\,p(\cdot\mid x)\right) = E_q\left[\ln\frac{q(\theta)}{p(\theta\mid x)}\right] = E_q\left[\ln q(\theta)\right] - E_q\left[\ln p(x,\theta)\right] + \ln p(x) ,
$$

using $p(\theta\mid x) = p(x,\theta)/p(x)$ and the fact that $\ln p(x)$ is constant with respect to $\theta$. Rearranging gives the identity, and since $D_{\text{KL}}\ge 0$ we get $\ln p(x)\ge\mathcal{L}(q)$ with equality iff $q = p(\cdot\mid x)$. $\blacksquare$

**Why this is useful.** $\ln p(x)$ is fixed by the data, so *maximizing the ELBO is minimizing the KL divergence to the posterior* — integration is replaced by optimization. Two standard recipes follow:

- **Mean-field.** Restrict to $q(\theta) = \prod_j q_j(\theta_j)$. Coordinate ascent gives the closed-form update $\ln q_j^\star(\theta_j) = E_{q_{-j}}\left[\ln p(x,\theta)\right] + \text{const}$, which for conditionally conjugate models stays inside the conjugate family.
- **Stochastic / amortized.** Parameterize $q_\phi$ by a neural network and optimize with reparameterized gradients, $\theta = \mu_\phi + \sigma_\phi\odot\epsilon$ with $\epsilon\sim\mathcal{N}(0,I)$. This is the variational autoencoder; its loss is exactly $-\mathcal{L}$, split as a reconstruction term plus $D_{\text{KL}}\left(q_\phi(\theta\mid x)\,\Vert\,p(\theta)\right)$.

**The known bias.** Because the objective is $D_{\text{KL}}(q\Vert p)$ rather than $D_{\text{KL}}(p\Vert q)$, the optimum is *mode-seeking*: $q$ pays an infinite penalty for placing mass where $p$ has none, but no penalty for missing mass. Mean-field variational posteriors are therefore systematically **too narrow**, and their credible intervals undercover. MCMC has the opposite trade-off — asymptotically exact but with no finite-time guarantee.

## 4. Computational & Algorithmic Insights

### 4.1 The Four Ways to Handle an Intractable Posterior

| Method | Idea | Cost | Exactness |
|---|---|---|---|
| Conjugacy | Closed-form update inside an exponential family | $O(n)$ | Exact, but restricted models |
| Laplace | Gaussian fit at the mode with covariance $\left[-\nabla^2\ln p\right]^{-1}$ | one optimization + one Hessian | Asymptotically exact; misses skew and multimodality |
| MCMC | Sample via a chain whose stationary law is the posterior | many likelihood evaluations | Exact in the limit; convergence unverifiable in finite time |
| Variational | Maximize the ELBO over a tractable family | one optimization | Biased (too narrow) but scalable and differentiable |

The practical rule: use conjugacy where the model allows it, MCMC when accuracy matters and dimension is moderate, variational methods when the model is a neural network, and Laplace whenever a trained point estimate already exists and cheap uncertainty is wanted.

### 4.2 MCMC Craft

- **Only ratios are needed.** Metropolis–Hastings uses $\tilde p(\theta')/\tilde p(\theta)$, so the intractable evidence cancels — this is the whole reason MCMC works.
- **Compute in log space.** Accept when $\ln u \lt \ln\tilde p(\theta') - \ln\tilde p(\theta) + \ln q(\theta\mid\theta') - \ln q(\theta'\mid\theta)$; exponentiating densities directly underflows immediately in realistic dimensions.
- **Gradient-based samplers.** Hamiltonian Monte Carlo and NUTS use $\nabla\ln p(\theta\mid x)$ to propose distant, high-acceptance moves; the random-walk Metropolis step size must scale like $d^{-1/2}$ whereas HMC scales like $d^{-1/4}$, which is why HMC dominates above a few dozen dimensions.
- **Diagnostics.** Report $\hat{R}$ (between- versus within-chain variance across multiple chains) and effective sample size $n_{\text{eff}} = n/\tau$ where $\tau$ is the integrated autocorrelation time. Trace plots detect stuck chains; divergences in HMC detect geometry the sampler cannot traverse.
- **Reparameterize hierarchies.** The "funnel" geometry of $\theta_j\sim\mathcal{N}(0,\tau^2)$ with $\tau$ unknown defeats samplers; the non-centered form $\theta_j = \tau z_j$, $z_j\sim\mathcal{N}(0,1)$ removes the pathological curvature.

### 4.3 Priors, Predictive Checks, and Model Comparison

- **Choosing priors.** Weakly informative beats flat: a $\mathcal{N}(0, 2.5^2)$ prior on standardized logistic-regression coefficients regularizes without asserting much, whereas an improper flat prior can make the posterior improper and silently break everything downstream.
- **Jeffreys' prior** $p_J(\theta)\propto\sqrt{\det I(\theta)}$ is the reparameterization-invariant default. For a Bernoulli rate it is $\text{Beta}(1/2,1/2)$; for a scale parameter it is $p(\sigma)\propto 1/\sigma$, i.e. flat in $\ln\sigma$.
- **Prior predictive checks.** Simulate $\theta\sim p(\theta)$ then $x\sim p(x\mid\theta)$ and look at the results. If the prior predicts effect sizes no experiment has ever produced, the prior is wrong regardless of how "uninformative" it was intended to be.
- **Posterior predictive checks.** Compare summary statistics of replicated data $x^{\text{rep}}\sim p(\cdot\mid x)$ with the observed statistics; systematic mismatch is model misspecification.
- **Occam's razor is automatic.** The evidence $p(x\mid M) = \int p(x\mid\theta,M)p(\theta\mid M)d\theta$ spreads a fixed unit of prior mass over parameter space, so a flexible model that could have predicted anything predicts each particular dataset weakly. No explicit complexity penalty is required; BIC is the leading asymptotic term of exactly this integral.

## 5. Real-World Physics & AI/ML Applications

### 5.1 AI / Machine Learning

- **Bayesian neural networks and deep ensembles.** Placing a prior over weights and approximating the posterior (via Laplace with a Kronecker-factored Fisher, SWAG, or MC-dropout) yields calibrated predictive uncertainty; deep ensembles are a crude but effective posterior sample.
- **Variational autoencoders.** The ELBO of Proof 3.7 is the training objective; the encoder is an amortized posterior $q_\phi(z\mid x)$ and the reparameterization trick makes the bound differentiable.
- **Gaussian processes.** Exact Bayesian inference over functions: the posterior mean is a kernel-weighted interpolation and the posterior variance grows away from the data, which drives Bayesian optimization's acquisition functions (expected improvement, UCB).
- **Thompson sampling.** For bandits and A/B tests, sample $\theta\sim p(\theta\mid x)$ and act greedily; the exploration rate is generated automatically by posterior width, and regret bounds are near-optimal.
- **Hierarchical models and partial pooling.** Per-group parameters drawn from a shared prior shrink noisy small-group estimates toward the population mean — the statistical content of multi-task learning, and the reason a player with 5 at-bats is not credited with a $0.600$ average.
- **Probabilistic programming.** Stan, PyMC, NumPyro, and Pyro let a model be written as a generative program and hand inference to HMC/NUTS or variational engines; conjugacy, the ELBO, and MCMC diagnostics are the machinery underneath.

### 5.2 Physics & Engineering

- **Kalman filtering and state estimation.** The measurement update is precisely the Normal–Normal conjugate step of Proof 3.3, with precisions adding; the prediction step propagates the prior through the dynamics. GPS, inertial navigation, and spacecraft attitude estimation are recursive Bayesian inference in production.
- **Gravitational-wave astronomy.** LIGO parameter estimation samples posteriors over masses, spins, and sky location using MCMC/nested sampling; detection claims are Bayes factors between signal and noise-only models.
- **Cosmological parameter estimation.** CMB and large-scale-structure analyses report posteriors over $\Omega_m$, $H_0$, and $\sigma_8$ with explicit priors, and tensions between datasets are diagnosed by comparing posteriors rather than point estimates.
- **Inverse problems and tomography.** Reconstruction from projections is ill-posed; a prior (smoothness, sparsity, total variation) selects among the infinitely many data-consistent solutions, and the posterior quantifies which features are actually resolved.
- **Experimental design and combining measurements.** Inverse-variance weighting of independent measurements is the Normal–Normal update; sequential Bayesian design picks the next measurement to maximize expected information gain, i.e. expected KL divergence between prior and posterior.
- **Reliability and rare-event risk.** With few or zero observed failures the MLE is $0$; a Gamma or Beta prior yields a finite, defensible failure-rate posterior with honest upper credible bounds — the standard in nuclear and aerospace safety analysis.

### 5.3 Key Formula Summary

| Object | Formula | Notes |
|---|---|---|
| Bayes' rule | $p(\theta \mid x) = p(x \mid \theta)p(\theta)/p(x)$ | Posterior $\propto$ likelihood $\times$ prior |
| Evidence | $p(x) = \int p(x \mid \theta)p(\theta)\,d\theta$ | Normalizer; also the model score |
| Posterior predictive | $p(\tilde{x} \mid x) = \int p(\tilde{x} \mid \theta)p(\theta \mid x)\,d\theta$ | Wider than plug-in |
| Beta–Binomial | $\text{Beta}(\alpha + k,\ \beta + n - k)$ | $\alpha,\beta$ are pseudo-counts |
| Normal–Normal | $\lambda_n = \lambda_0 + n\lambda$, $\mu_n = (\lambda_0\mu_0 + n\lambda\bar{x})/\lambda_n$ | Precisions add |
| Gamma–Poisson | $\text{Gamma}\left(a + \sum_i x_i,\ b + n\right)$ | Predictive is negative binomial |
| Dirichlet–Categorical | $\text{Dirichlet}(\alpha + \text{counts})$ | Additive smoothing |
| Bayes estimators | mean / median / mode for $L_2$ / $L_1$ / $0$–$1$ loss | Loss picks the summary |
| Bayes factor | $\text{BF}_{12} = p(x \mid M_1)/p(x \mid M_2)$ | Automatic Occam penalty |
| Jeffreys prior | $p_J(\theta) \propto \sqrt{\det I(\theta)}$ | Reparameterization-invariant |
| Bernstein–von Mises | $p(\theta \mid x) \to \mathcal{N}\left(\hat\theta, (nI)^{-1}\right)$ | Prior washes out at $O(1/n)$ |
| ELBO | $\ln p(x) = \mathcal{L}(q) + D_{\text{KL}}\left(q \Vert p(\theta \mid x)\right)$ | Integration becomes optimization |
| Variance split | $\operatorname{Var}(\tilde{x} \mid x) = E[\operatorname{Var}] + \operatorname{Var}(E)$ | Aleatoric plus epistemic |

## 6. Canonical Literature Mapping & References

| Source | Chapters / Sections | Coverage |
|---|---|---|
| Gelman et al., *Bayesian Data Analysis* (3rd ed.) | Chapters 1–5 | Conjugate models, hierarchical models, predictive checks — the standard reference. |
| Wasserman, *All of Statistics* | Chapter 11 | Compact, critical comparison of Bayesian and frequentist inference. |
| Casella & Berger, *Statistical Inference* (2nd ed.) | Sections 7.2.3, 7.3.4 | Bayes estimators and decision-theoretic risk. |
| Bishop, *Pattern Recognition and Machine Learning* | Chapters 2–3, 10 | Conjugate priors, Bayesian linear regression, variational inference. |
| Murphy, *Probabilistic ML: Advanced Topics* | Chapters 3–4, 7, 12 | Modern algorithmic treatment: MCMC, VI, Bayesian deep learning. |
| MacKay, *Information Theory, Inference, and Learning Algorithms* | Chapters 2–3, 28–29 | Occam's razor via evidence; Monte Carlo methods with unusual clarity. |
| Robert, *The Bayesian Choice* (2nd ed.) | Chapters 2–4 | Decision theory, noninformative priors, admissibility. |
| van der Vaart, *Asymptotic Statistics* | Chapter 10 | Bernstein–von Mises at full rigor. |
| Rasmussen & Williams, *Gaussian Processes for Machine Learning* | Chapters 2–5 | Exact Bayesian inference over functions. |

**Reading path**: Gelman et al. Chs. 1–3 for conjugate models and the Bayesian workflow, MacKay Chs. 28–29 for the evidence and Occam's razor, Bishop Ch. 10 for variational inference, Murphy's advanced volume for the algorithms, and van der Vaart Ch. 10 for the asymptotic bridge back to Topic 09.